In [2]:
pip install torch 

   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/114.6 MB ? eta -:--:--
   ---------------------------------------- 1.0/114.6 MB 3.9 MB/s eta 0:00:30
    --------------------------------------- 1.8/114.6 MB 4.2 MB/s eta 0:00:27
    --------------------------------------- 1.8/114.6 MB 4.2 MB/s eta 0:00:27
    --------------------------------------- 2.4/114.6 MB 2.6 MB/s eta 0:00:43
    --------------------------------------- 2.6/114.6 MB 2.8 MB/s eta 0:00:40
    --------------------------------------- 2.6/114.6 MB 2.8 MB/s eta 0:00:40
    --------------------------------------- 2.6/114.6 MB 2.8 MB/s eta 0:00:40
    --------------------------------------- 2.6/114.6 MB 2.8 MB/s eta 0:00:40
    --------------------------------------- 2.6/114.6 MB 2.8 MB/s eta 0:00:40
    --------------------------------------- 2.6/114.6 MB 2.8 MB/s eta 0:00:40
   - -------------------------------------- 2.9/114.6 MB 1.2 MB/s eta 0:01:31


In [12]:
import torch
import torch.nn as nn
import math

class InputEmbeddings(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

In [13]:
d_model = 8
vocab_size = 1000
embed = InputEmbeddings(d_model, vocab_size)
sentence_tokens = torch.tensor([[10, 25, 500, 30, 31, 85]])
output = embed(sentence_tokens)
print(output.shape)

torch.Size([1, 6, 8])


In [14]:
print(output)

tensor([[[-3.4131,  2.5122,  2.8332, -1.4880, -2.5007,  3.9291,  3.4303,
          -0.4674],
         [-1.3095, -1.6754, -1.4405,  0.8656,  2.1557, -1.5837,  0.5291,
          -7.7858],
         [ 1.0644, -1.4326, -1.9271, -2.0246,  1.7704, -5.7622, -1.3493,
           2.5835],
         [-0.2036, -2.3440, -0.8345,  0.2039,  2.5706,  4.0155, -3.4957,
           0.5361],
         [10.2856, -1.5862, -2.4240, -2.3865, -0.2110,  0.6036, -2.9462,
           2.0139],
         [ 0.9163, -2.0577, -2.1357, -1.6084, -1.1799, -1.2828, -5.3669,
          -1.2173]]], grad_fn=<MulBackward0>)


In [19]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, seq_len: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout)
        self.seq_length = seq_len
        pe = torch.zeros(seq_len, d_model)

        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [20]:
d_model = 8
seq_len = 1000
pos_enc = PositionalEncoding(d_model, seq_len, dropout=0.1)
x = embed(sentence_tokens)
output = pos_enc(x)
print(pos_enc.pe.shape)
print(output)

torch.Size([1, 1000, 8])
tensor([[[-3.7923,  3.9025,  3.1481, -0.0000, -2.7785,  5.4768,  3.8114,
           0.5918],
         [-0.5200, -1.2612, -1.4897,  2.0674,  2.4064, -0.6486,  0.5890,
          -0.0000],
         [ 0.0000, -2.0542, -1.9205, -1.1606,  1.9894, -5.2915, -1.4970,
           0.0000],
         [-0.0694, -3.7044, -0.5988,  1.2880,  2.8896,  5.5723, -3.8808,
           1.7068],
         [ 0.0000, -2.4887, -2.2606, -1.6282, -0.1900,  1.7809, -3.2691,
           3.3488],
         [-0.0473, -1.9711, -1.8403, -0.8120, -1.2555, -0.3157, -5.9577,
          -0.2415]]], grad_fn=<MulBackward0>)
